In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import json

os.chdir(next(p for p in Path.cwd().parents if (p / 'src').exists()))
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 

env_path = sys.prefix
os.environ['CC'] = f"{env_path}/bin/x86_64-conda-linux-gnu-cc"
os.environ['CXX'] = f"{env_path}/bin/x86_64-conda-linux-gnu-c++"

os.environ['PKG_CONFIG_PATH'] = f"{env_path}/lib/pkgconfig"
os.environ['LD_LIBRARY_PATH'] = f"{env_path}/lib"

from src.data.fem_generator import generate_fem_data, generate_parametric_fem_data, generate_noise

In [2]:
# Generation for 01_pipeline_evolution/02_fixed_scar
N_GRID = 400
R_A = 1.0 / N_GRID

base_dir = os.path.join('data', 'fixed_scar')
dataset_path = f'{base_dir}/dataset.npz'
grid_path = f'{base_dir}/fem_grid.npy'

if not os.path.exists(dataset_path):
    print("[DATA] Data not found. Starting FEM generation for fixed scar")
    generate_fem_data(N=N_GRID, r_a=R_A, base_dir=base_dir)
else:
    print(f"[DATA] FEM data already present in {dataset_path}")

if not os.path.exists(grid_path):
    print("[DATA] Data not found. Starting FEM generation for fixed scar")
    generate_fem_data(N=N_GRID, r_a=R_A, base_dir=base_dir)
else:
    print(f"[DATA] FEM data already present in {grid_path}")

[DATA] FEM data already present in data/fixed_scar/dataset.npz
[DATA] FEM data already present in data/fixed_scar/fem_grid.npy


In [3]:
# Generation for 01_pipeline_evolution/03_parametric_pinn and 01_pipeline_evolution/04_uq-vae
N_GRID = 400
R_A = 1.0 / N_GRID

resolutions = [400]
base_dir = os.path.join('data', 'parametric')

check_file = f'{base_dir}/{resolutions[-1]}x{resolutions[-1]}/test.npz'

if not os.path.exists(check_file):
    print("[DATA] Data not found. Starting FEM generation...")
    base_dir, time_per_res = generate_parametric_fem_data(
        resolutions=resolutions, n_sim=1000, n_points_save=1020, r_a=R_A,
        save_shared_data=False, save_grid=True
    )
    data_generated = True
else:
    data_generated = False
    print(f"[DATA] FEM data already present in {check_file}")

noise_file = f'{base_dir}/noise.npz'
noise_ok = os.path.exists(noise_file) and 'noise_std_physical' in np.load(noise_file).files

if data_generated or not noise_ok:
    print("[NOISE] Generating 2% relative noise...")
    generate_noise(resolutions=resolutions, base_dir=base_dir, noise_level_relative=0.02, seed=42)
else:
    print("[NOISE] noise.npz already present.")

[DATA] FEM data already present in data/parametric/400x400/test.npz
[NOISE] Generating 2% relative noise...
Reference grid: 400x400
Signal scale (max|values_clean|): 0.071223
Noise std (physical, 2.0% of signal): 0.001424
[OK] Saved data/parametric/noise.npz
[OK] Noise updated in data/parametric/400x400/train.npz
[OK] Noise updated in data/parametric/400x400/val.npz
[OK] Noise updated in data/parametric/400x400/test.npz


In [4]:
# Generation for 02_error_analysis/ 
resolutions = [100, 200, 400, 800]
R_A = 1.0 / resolutions[0]

base_data_dir = os.path.join('data', 'multiresolution')
times_file = os.path.join(base_data_dir, 'generation_times.json')

os.makedirs(base_data_dir, exist_ok=True)
# times
if os.path.exists(times_file):
    with open(times_file, 'r') as f:
        time_per_res = {int(k): float(v) for k, v in json.load(f).items()}
else:
    time_per_res = {}

missing_resolutions = []
for N in resolutions:
    res_test_file = os.path.join(base_data_dir, f"{N}x{N}", "test.npz")
    if not os.path.exists(res_test_file):
        missing_resolutions.append(N)
    else:
        res_dir = os.path.join(base_data_dir, f"{N}x{N}")
        print(f"[DATA] FEM data already present in: {res_dir}")

data_generated = False
if missing_resolutions:
    print(f"[DATA] Data not found. Starting FEM generation.")
    _, new_times = generate_parametric_fem_data(
        resolutions=missing_resolutions, n_sim=1000, n_points_save=1020, r_a=R_A,
        base_dir=base_data_dir, save_shared_data=True
    )
    time_per_res.update(new_times)
    data_generated = True

noise_file = os.path.join(base_data_dir, 'noise.npz')
noise_ok = os.path.exists(noise_file) and 'noise_std_physical' in np.load(noise_file).files

if data_generated or not noise_ok:
    print("[NOISE] Generating 2% relative noise...")
    generate_noise(resolutions=resolutions, base_dir=base_data_dir)
else:
    print("[NOISE] noise.npz already present.")

[DATA] FEM data already present in: data/multiresolution/100x100
[DATA] FEM data already present in: data/multiresolution/200x200
[DATA] FEM data already present in: data/multiresolution/400x400
[DATA] FEM data already present in: data/multiresolution/800x800
[NOISE] Generating 2% relative noise...
Reference grid: 800x800
Signal scale (max|values_clean|): 0.066552
Noise std (physical, 2.0% of signal): 0.001331
[OK] Saved data/multiresolution/noise.npz
[OK] Noise updated in data/multiresolution/100x100/train.npz
[OK] Noise updated in data/multiresolution/200x200/train.npz
[OK] Noise updated in data/multiresolution/400x400/train.npz
[OK] Noise updated in data/multiresolution/800x800/train.npz
[OK] Noise updated in data/multiresolution/100x100/val.npz
[OK] Noise updated in data/multiresolution/200x200/val.npz
[OK] Noise updated in data/multiresolution/400x400/val.npz
[OK] Noise updated in data/multiresolution/800x800/val.npz
[OK] Noise updated in data/multiresolution/100x100/test.npz
[OK]